In [0]:
from pyspark.sql import functions as F

catalog = spark.sql(
    "SELECT current_catalog() AS catalog"
).first()["catalog"]

laps_df = spark.table(
    f"{catalog}.f1_silver.lap_times"
)

pits_df = spark.table(
    f"{catalog}.f1_silver.pit_stops"
)

before_lap_df = laps_df.select(
    "race_id",
    "driver_id",
    F.col("lap_number").alias("reference_lap"),
    F.col("position").alias("position_before")
)

after_lap_df = laps_df.select(
    "race_id",
    "driver_id",
    F.col("lap_number").alias("reference_lap"),
    F.col("position").alias("position_after")
)

In [0]:
pit_impact_df = (
    pits_df.alias("p")
    .join(
        before_lap_df.alias("b"),
        (
            (F.col("p.race_id") == F.col("b.race_id"))
            & (F.col("p.driver_id") == F.col("b.driver_id"))
            & (F.col("b.reference_lap") == F.col("p.pit_lap") - 1)
        ),
        "left"
    )
    .join(
        after_lap_df.alias("a"),
        (
            (F.col("p.race_id") == F.col("a.race_id"))
            & (F.col("p.driver_id") == F.col("a.driver_id"))
            & (F.col("a.reference_lap") == F.col("p.pit_lap") + 1)
        ),
        "left"
    )
    .select(
        F.col("p.race_id"),
        F.col("p.driver_id"),
        F.col("p.stop_number"),
        F.col("p.pit_lap"),
        F.col("p.duration_ms"),
        F.col("b.position_before"),
        F.col("a.position_after")
    )
    .withColumn(
        "positions_gained",
        F.col("position_before") - F.col("position_after")
    )
)

display(pit_impact_df)

In [0]:
pit_impact_table = f"{catalog}.f1_gold.pit_stop_impact"

(
    pit_impact_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(pit_impact_table)
)

print(f"Saved: {pit_impact_table}")